# 🛡️ CIDA v4 Plan B: Anti-Hallucination Verifier

**Concept Pivot:** Instead of generating Chain-of-Thought planning tokens, CIDA acts as an independent auditor to verify LLM responses and detect hallucinations.

**Mechanism:**
1. Input: Prompt + LLM Response -> Frozen Embedding Model.
2. CIDA Agents: Prosecutor, Defender, Skeptic, and Integrator debate the response validity.
3. Output: Factual validation score (0.0 to 1.0) and uncertainty metrics derived from agent disagreement.

## 0 · Setup

In [ ]:
!pip install -q torch transformers datasets accelerate torchdiffeq bitsandbytes
import subprocess, sys, os
if not os.path.exists('CIDA-plugin'):
    subprocess.run(['git','clone','--depth','1',
                   'https://github.com/Kairatzh/CIDA-plugin.git'], check=True)
sys.path.insert(0, os.path.abspath('CIDA-plugin'))
from google.colab import drive
drive.mount('/content/drive')
print('Setup complete.')


## 1 · Dataset (HaluEval - QA & Summarization Hallucinations)

In [ ]:
from datasets import load_dataset
import random
random.seed(42)

print('Loading HaluEval dataset...')
try:
    # Load QA subset of HaluEval
    dataset = load_dataset('jiazhengli/HaluEval', 'qa', split='data')
    all_samples = []
    for ex in dataset:
        q = ex['question']
        # Positive sample (Correct)
        all_samples.append({'question': q, 'response': ex['right_answer'], 'label': 1})
        # Negative sample (Hallucinated)
        all_samples.append({'question': q, 'response': ex['hallucinated_answer'], 'label': 0})
except Exception as e:
    print(f'HaluEval failed: {e}, falling back to synthetic verification data')
    all_samples = [
        {'question': 'What is the capital of France?', 'response': 'Paris is the capital of France.', 'label': 1},
        {'question': 'What is the capital of France?', 'response': 'London is the capital of France.', 'label': 0},
    ] * 500

random.shuffle(all_samples)
all_samples = all_samples[:20000]  # Take 20K samples (10K QA pairs)
split = int(len(all_samples) * 0.9)
train_data, val_data = all_samples[:split], all_samples[split:]
print(f'Train size: {len(train_data)} | Val size: {len(val_data)}')


## 2 · Frozen Sentence Encoder (bge-large-en-v1.5)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

MODEL_NAME = 'BAAI/bge-large-en-v1.5'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to('cuda')
for p in encoder.parameters(): p.requires_grad = False
encoder.eval()

D_EMBED = encoder.config.hidden_size
print(f'Encoder loaded. Embedding dimension: {D_EMBED}')


## 3 · Cache Encoded Representations

In [ ]:
from tqdm.auto import tqdm
import os

def cache_verifications(data, name):
    os.makedirs('cache_verifier', exist_ok=True)
    cached = []
    for sample in tqdm(data, desc=f'Caching {name}'):
        # Combine question and response into auditor format
        text = f"Question: {sample['question']}\nResponse: {sample['response']}"
        enc = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
        
        with torch.no_grad():
            out = encoder(**enc)
            # Mean pooling
            emb = out.last_hidden_state.mean(dim=1)[0].cpu().half()
            
        cached.append({'embedding': emb, 'label': sample['label']})
        
    path = f'cache_verifier/{name}_cached.pt'
    torch.save(cached, path)
    return path

print('Caching Train embeddings...')
train_path = cache_verifications(train_data, 'train')
print('Caching Val embeddings...')
val_path = cache_verifications(val_data, 'val')


## 4 · CIDA Verifier Configuration (Binary Classifier)

In [ ]:
from cida_plugin.config import CIDAPluginConfig
from cida_plugin.core import CIDAPlugin
import torch.nn as nn

cfg = CIDAPluginConfig(
    d_input=D_EMBED,
    d_hidden=512,
    d_message=512,
    num_agents=4,
    num_classes=2,  # 0 = Hallucination, 1 = Factual
    max_rounds=4,
    num_attn_heads=8,
    use_perspective_projector=True,
    anonymous_messages=True,
    sparse_comm_k=3,
    use_dynamic_trust=True
)

cida = CIDAPlugin(cfg).to('cuda').float()
print('CIDA Verifier initialized successfully.')


## 5 · Training CIDA Verifier

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

class EmbedDataset(Dataset):
    def __init__(self, path):
        self.data = torch.load(path)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]['embedding'].float(), self.data[idx]['label']

train_loader = DataLoader(EmbedDataset(train_path), batch_size=64, shuffle=True)
val_loader = DataLoader(EmbedDataset(val_path), batch_size=64, shuffle=False)

optimizer = torch.optim.AdamW(cida.parameters(), lr=1e-3, weight_decay=0.01)
epochs = 5

for epoch in range(epochs):
    cida.train()
    total_loss, correct, total = 0, 0, 0
    for embeds, labels in train_loader:
        embeds, labels = embeds.to('cuda'), labels.to('cuda')
        optimizer.zero_grad()
        
        out = cida(pooled=embeds)
        logits = out['p_final']  # Consensus probability distribution
        loss_ce = F.cross_entropy(logits, labels)
        loss_orth = out['L_orth']
        
        loss = loss_ce + 0.1 * loss_orth
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * embeds.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += embeds.size(0)
        
    train_acc = correct / total
    train_loss = total_loss / total
    
    # Validation
    cida.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for embeds, labels in val_loader:
            embeds, labels = embeds.to('cuda'), labels.to('cuda')
            out = cida(pooled=embeds)
            logits = out['p_final']
            val_loss += F.cross_entropy(logits, labels).item() * embeds.size(0)
            preds = logits.argmax(dim=-1)
            val_correct += (preds == labels).sum().item()
            val_total += embeds.size(0)
            
    print(f'Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss/val_total:.4f} | Val Acc: {val_correct/val_total*100:.2f}%')

# Save Weights
save_dir = 'cida_verifier_weights'
os.makedirs(save_dir, exist_ok=True)
cfg.save_pretrained(save_dir)
torch.save(cida.state_dict(), f'{save_dir}/pytorch_model.bin')
print(f'Verifier weights saved to {save_dir}/')


## 6 · Backup to Google Drive

In [ ]:
import shutil
dst = '/content/drive/MyDrive/cida_verifier_weights'
os.makedirs(dst, exist_ok=True)
for f in os.listdir(save_dir):
    shutil.copy2(f'{save_dir}/{f}', dst)
print(f'Backed up to Google Drive: {dst}')
